In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
spark=SparkSession.builder.appName("rdd_inv").getOrCreate()

wal_data = [
    (1,'2020-01-09','success', 70.59, 6.56,14.36),
    (2,'2020-01-24','success', 93.36, 22.68,19.9),
    (3,'2020-02-08','fail', 51.24, 11.39,21.32),
    (4,'2020-02-23','success', 61.58,8.04,44.26),
    (5,'2020-03-09','success', 25.04,7.19,1.74),
    (6,'2020-03-24','fail', 45.57, 4.68,24.19),
    (7,'2020-04-08','success', 24.45,12.69,15.91),
    (8,'2020-04-23','success', 48.22,11.2,48.82),
    (9,'2020-05-08','success', 56.63,4.04,16.08),
    (10,'2020-05-23','fail', 19.03,16.65,11.22),
    (11,'2020-06-07','fail', 81,6.56,26.6),
    (12,'2020-06-22','fail', 21.32,8.86,28.57),
    (13,'2020-07-07','fail', 14.74,17.76,19.33),
    (14,'2020-07-22','success',66.73,13.68,14.07),
    (15,'2020-08-06','success',32.98,16.17,25.34),
    (16,'2020-08-21','success',46.49,1.84,41.9),
    (17,'2020-09-05','fail', 45.98,12.2,2.46),
    (18,'2020-09-20','success',3.14,24.8,36.6),
    (19,'2020-10-05','success',75.33,23.04,29.99),
    (20,'2020-10-20','success', 53.76,22.94,18.74)
]

cust_schema = StructType([
    StructField("request_id", IntegerType(), True),
    StructField("request_date", StringType(), True),
    StructField("request_status", StringType(), True),
    StructField("distance_to_travel", DoubleType(), True),
    StructField("monetary_cost", DoubleType(), True),
    StructField("driver_to_client_distance", DoubleType(), True)
])

df = spark.createDataFrame(wal_data, schema=cust_schema)
df2=df.withColumn("trans_request_date",try_to_date(col("request_date"),"yyyy-MM-dd"))
df2=df2.withColumn("trans_request_month_yr",date_format(col("trans_request_date"),"yyyy-MM"))
df2=df2.withColumn("distance_per_dollar",round(col("distance_to_travel")/col("monetary_cost"),2))

monthly_sum = df2.groupBy("trans_request_month_yr").agg(
    round(sum("distance_to_travel"),2).alias("total_distance"),
    round(sum("monetary_cost"),2).alias("total_cost")
).orderBy("trans_request_month_yr")
window_spec = Window.orderBy(col("trans_request_month_yr"))

df3 = df2.alias("t1").join(
    monthly_sum.alias("t2"),
    on="trans_request_month_yr",
    how="inner"
)
df3 = df3.withColumn(
    "prev_amount",
    lag(
        round(col("distance_to_travel") / col("monetary_cost"), 2)
    ).over(window_spec)
)
df4=df3.select(sqrt(mean(col("prev_amount")-col("distance_per_dollar"))**2))
df4.display()




















